# Create Hosted Agent From Code — `@azure/ai-projects`

This notebook demonstrates uploading a code zip as a new version of a code-based Hosted Agent, polling for provisioning, and downloading it back to verify the round-trip.

It mirrors the [`createHostedAgentFromCode.ts`](./createHostedAgentFromCode.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

The dependency resolution mode is selected via the `FOUNDRY_HOSTED_AGENT_REMOTE_BUILD` environment variable (default: `false`):

- `false` (bundled) — uploads `assets/responses-echo-agent.zip`, which bundles the agent source plus pre-built dependencies so the service skips dependency installation entirely.
- `true` (remote_build) — uploads the same zip but instructs the service to resolve dependencies remotely from the manifest included in the zip.

The agent must already exist; create it first with the [`createHostedAgentFromImage`](./createHostedAgentFromImage.ipynb) sample.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_HOSTED_AGENT_NAME`, `FOUNDRY_HOSTED_AGENT_REMOTE_BUILD` (optional; defaults to `false`).

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  CodeDependencyResolution,
  CreateAgentVersionFromCodeContent,
  HostedAgentDefinition,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";
import { createHash } from "node:crypto";
import { readFileSync, writeFileSync } from "node:fs";
import path from "node:path";
import { buffer } from "node:stream/consumers";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const agentName = process.env["FOUNDRY_HOSTED_AGENT_NAME"] ?? "<hosted agent name>";
const useRemoteBuild =
  (process.env["FOUNDRY_HOSTED_AGENT_REMOTE_BUILD"] ?? "false").trim().toLowerCase() === "true";

const codeZipPath = path.resolve("../assets/responses-echo-agent.zip");

function sha256Hex(data: Uint8Array): string {
  return createHash("sha256").update(data).digest("hex");
}

console.log(`Agent: ${agentName}`);
console.log(`Remote build: ${useRemoteBuild}`);

Agent: MyTestHostedAgent5
Remote build: false
Remote build: false


In [6]:
// Create the AI Project client
const project: any = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [3]:
// Create a new version on the existing agent from code
const dependencyResolution: CodeDependencyResolution = useRemoteBuild ? "remote_build" : "bundled";

const codeZip = readFileSync(codeZipPath);
const codeZipSha256 = sha256Hex(codeZip);

const definition: HostedAgentDefinition = {
  kind: "hosted",
  cpu: "0.5",
  memory: "1Gi",
  protocol_versions: [{ protocol: "responses", version: "1.0.0" }],
  code_configuration: {
    runtime: "python_3_14",
    entry_point: ["python", "main.py"],
    dependency_resolution: dependencyResolution,
  },
};

const content: CreateAgentVersionFromCodeContent = {
  metadata: {
    description: `Code-based hosted agent uploaded with dependency_resolution=${dependencyResolution}.`,
    definition,
  },
  code: { contents: codeZip, contentType: "application/zip", filename: "code.zip" },
};

console.log(`Creating code-based agent version (dependency_resolution=${dependencyResolution})...`);
const created = await project.agents.createVersionFromCode(agentName, codeZipSha256, content);
const createdVersion = created.version;
console.log(`Created code-based hosted agent version: ${createdVersion}`);

Creating code-based agent version (dependency_resolution=bundled)...
Created code-based hosted agent version: 4


In [4]:
// Poll until the agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, createdVersion);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1}/60)`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: active (attempt 1/60)


In [7]:
// Download the code for the version we just created and verify the round-trip
console.log("Downloading agent version code...");
const downloadResult = await project.agents.downloadAgentCode(agentName, {
  agentVersion: createdVersion,
});

const downloadedBytes = downloadResult.readableStreamBody
  ? new Uint8Array(await buffer(downloadResult.readableStreamBody))
  : downloadResult.blobBody
    ? new Uint8Array(await (await downloadResult.blobBody).arrayBuffer())
    : undefined;

if (downloadedBytes) {
  const downloadedSha256 = sha256Hex(downloadedBytes);
  const downloadPath = path.resolve(`${agentName}-${createdVersion}.zip`);
  writeFileSync(downloadPath, downloadedBytes);
  console.log(
    `Downloaded version code zip to ${downloadPath}: ${downloadedBytes.length} bytes, ` +
      `sha256=${downloadedSha256} (matches uploaded: ${downloadedSha256 === codeZipSha256})`,
  );
} else {
  console.warn("No content found in the downloaded agent code.");
}

console.log("\nSample completed!");

Downloaded version code zip to c:\Users\bogou\projects\azure-sdk-for-js\sdk\ai\ai-projects\samples-dev\agents\hostedAgents\MyTestHostedAgent5-4.zip: 1416 bytes, sha256=895a5fbae4b2977f1fd60abeb054d5ee5730d12ff19ec0124460709997a2ff2c (matches uploaded: true)

Sample completed!
